# Injury Severity Analysis — NCDB 2019–2020

Loading and cleaning is in `01_load_and_clean.ipynb`. This notebook does the analysis.

The database holds 471,046 records — one row per person involved in a police-reported
collision in Canada in 2019 or 2020.

Two questions:

1. Who gets hurt worst, people inside a vehicle or outside it?
2. Does restraint use change the odds of injury once age, weather and year are accounted for?

## Load the repo

Colab sessions start empty and are wiped on disconnect, so the repo is cloned
into the session each time. Everything below runs from the project folder.

In [1]:
if (!dir.exists("/content/data-science-portfolio")) {
  system("git clone https://github.com/lautrevor/data-science-portfolio.git /content/data-science-portfolio")
}
setwd("/content/data-science-portfolio/project-3-bc-collision-analysis")

getwd()
list.files("sql")

[1] "/content/data-science-portfolio/project-3-bc-collision-analysis"

[1] "01_road_user.sql" "collisions.db"

## Setup

SQL queries are kept as separate files in `sql/` rather than inline, so each one can be read
on its own. `run()` reads one of those files and executes it against the database, returning
an R data frame.

The row count below confirms the connection works and the table is complete.

In [2]:
install.packages(c("DBI","RSQLite","dplyr","broom","ggplot2"))
library(DBI); library(RSQLite); library(dplyr); library(broom); library(ggplot2)

con <- dbConnect(SQLite(), "sql/collisions.db")
run <- function(path) dbGetQuery(con, paste(readLines(path), collapse = "\n"))

dbGetQuery(con, "SELECT COUNT(*) AS n FROM collisions")

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




n
<int>
471046


## Question 1 — who gets hurt worst?

Rate per person involved, not per collision. `P_USER` codes: 1 driver, 2 passenger,
3 pedestrian, 4 cyclist, 5 motorcyclist. Rows where injury severity was not recorded
are excluded; the count is reported below.

In [3]:
writeLines('SELECT
  P_USER,
  COUNT(*) AS persons,
  ROUND(100.0 * SUM(CASE WHEN P_ISEV = 3      THEN 1 ELSE 0 END) / COUNT(*), 2) AS fatality_pct,
  ROUND(100.0 * SUM(CASE WHEN P_ISEV IN (2,3) THEN 1 ELSE 0 END) / COUNT(*), 2) AS harmed_pct
FROM collisions
WHERE P_ISEV IN (1,2,3)
  AND P_USER IN (1,2,3,4,5)
GROUP BY P_USER
ORDER BY fatality_pct DESC;', "sql/01_road_user.sql")

In [4]:
run("sql/01_road_user.sql")

P_USER,persons,fatality_pct,harmed_pct
<dbl>,<int>,<dbl>,<dbl>
5,11383,3.77,94.96
3,18950,3.17,97.86
4,8989,0.99,96.25
1,280893,0.61,53.45
2,102267,0.55,53.42


Motorcyclists are killed at 3.77% and pedestrians at 3.17%, roughly six and five times
the driver rate of 0.61%.

The injury column is the wider gap. Around 95–98% of people outside a vehicle are
injured or killed, against 53% of drivers and passengers. Half of everyone inside a
vehicle walks away from a reported collision. Almost nobody outside one does.

Cyclists split the difference: